# cross-model comparison

logistic (sq2) vs RF + xgb (sq3) vs the embedding NN, all on the shared holdout. unified roc / calibration / shap.

first part re-runs the shared starter so this stands alone.

_Adam_ (synthesis); models from _Nick_, _Jordan_, _Adam_

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

RANDOM_STATE = 42  # shared seed for reproducibility

KAGGLE_DIR = Path("/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted")
LOCAL_DIR = Path("cleaned")

if KAGGLE_DIR.exists():
    CLEAN_DIR = KAGGLE_DIR
    ENV = "Kaggle"
else:
    CLEAN_DIR = LOCAL_DIR
    ENV = "Local"

print(f"Environment: {ENV}")
print(f"Data directory: {CLEAN_DIR}")

In [ ]:
# Columns from the pooled file that all SQs may need
POOLED_COLS = [
    "RECENT_CHECKUP",       # 1 = checkup within past year
    "GOT_FLUSHOT",          # 1 = flu shot in past 12 months
    "HAS_EXERCISE",         # 1 = physical activity in past 30 days
    "_RFSMOK3",             # 1 = not a current smoker, 2 = current smoker

    "_AGE_G",               # age group (1-6)
    "_SEX",                 # 1 = male, 2 = female
    "_IMPRACE",             # imputed race/ethnicity (1-6)
    "MARITAL",              # marital status (1-6)

    "_EDUCAG",              # education level (1-4)
    "EMPLOY1",              # employment status (1-8)

    "_BMI5_SCALED",         # BMI (continuous, already divided by 100)
    "HAS_DEPRESSION",       # 1 = told had depressive disorder
    "ADDEPEV3",             # depression raw (for alternate encoding)

    "HAS_PERSONAL_DOCTOR",  # 1 = has personal doctor

    "STATE_FIPS",           # state FIPS code

    "sample_weight",        # normalized pooled weight for model fitting
    "SURVEY_YEAR",          # source year (2020-2024)
]

df = pd.read_parquet(CLEAN_DIR / "brfss_2020_2024_pooled_ml.parquet", columns=POOLED_COLS)

# Downcast floats to save memory
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

print(f"Pooled dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nYear distribution:")
print(df["SURVEY_YEAR"].value_counts().sort_index())

In [ ]:

income_parts = []

# 2020: _INCOMG already has the 5-level scheme
yr_df = pd.read_parquet(CLEAN_DIR / "brfss_2020_ml.parquet", columns=["_INCOMG"])
yr_df.rename(columns={"_INCOMG": "INCOME_GROUP"}, inplace=True)
income_parts.append(yr_df)

# 2021-2024: _INCOMG1 needs collapsing
for year in [2021, 2022, 2023, 2024]:
    yr_df = pd.read_parquet(CLEAN_DIR / f"brfss_{year}_ml.parquet", columns=["_INCOMG1"])
    # Collapse upper income brackets: 5,6,7 -> 5
    yr_df["INCOME_GROUP"] = yr_df["_INCOMG1"].where(yr_df["_INCOMG1"] <= 4, 5.0)
    yr_df.drop(columns=["_INCOMG1"], inplace=True)
    income_parts.append(yr_df)

income_col = pd.concat(income_parts, ignore_index=True)
df["INCOME_GROUP"] = income_col["INCOME_GROUP"].values
del income_parts, income_col

print("INCOME_GROUP value counts:")
print(df["INCOME_GROUP"].value_counts(dropna=False).sort_index())
print(f"\nLabels: 1=<$15k, 2=$15-25k, 3=$25-35k, 4=$35-50k, 5=$50k+")

In [ ]:

access_parts = []

for year in [2020, 2021, 2022, 2023, 2024]:
    yr_path = CLEAN_DIR / f"brfss_{year}_ml.parquet"

    # Determine which columns to load from this year
    load_cols = []
    if year in (2020, 2024):
        load_cols.append("HAS_HEALTH_PLAN")
    if year in (2021, 2022, 2023, 2024):
        load_cols.append("COST_BARRIER")

    # 2020 has MEDCOST (raw) instead of COST_BARRIER — derive it
    if year == 2020:
        load_cols.append("MEDCOST")

    yr_df = pd.read_parquet(yr_path, columns=load_cols)

    # Derive COST_BARRIER for 2020 from MEDCOST (1=Yes, 2=No, 7=DK, 9=Ref)
    if year == 2020:
        yr_df["MEDCOST"] = yr_df["MEDCOST"].replace({7.0: np.nan, 9.0: np.nan})
        yr_df["COST_BARRIER"] = np.where(
            yr_df["MEDCOST"].isna(), np.nan,
            np.where(yr_df["MEDCOST"] == 1.0, 1.0, 0.0),
        )
        yr_df.drop(columns=["MEDCOST"], inplace=True)

    # Fill missing columns with NaN (variable not asked that year)
    for col in ["HAS_HEALTH_PLAN", "COST_BARRIER"]:
        if col not in yr_df.columns:
            yr_df[col] = np.nan

    access_parts.append(yr_df[["HAS_HEALTH_PLAN", "COST_BARRIER"]])

access_df = pd.concat(access_parts, ignore_index=True)
df["HAS_HEALTH_PLAN"] = access_df["HAS_HEALTH_PLAN"].values
df["COST_BARRIER"] = access_df["COST_BARRIER"].values
del access_parts, access_df

df["HAS_HEALTH_PLAN_MISSING"] = df["HAS_HEALTH_PLAN"].isna().astype("float32")
df["COST_BARRIER_MISSING"] = df["COST_BARRIER"].isna().astype("float32")

print("HAS_HEALTH_PLAN availability by year:")
print(df.groupby("SURVEY_YEAR")["HAS_HEALTH_PLAN"].apply(lambda s: s.notna().mean()).round(3))
print("\nCOST_BARRIER availability by year:")
print(df.groupby("SURVEY_YEAR")["COST_BARRIER"].apply(lambda s: s.notna().mean()).round(3))

In [ ]:
df["NON_SMOKER"] = np.where(
    df["_RFSMOK3"].isna(), np.nan,
    np.where(df["_RFSMOK3"] == 1.0, 1.0, 0.0),
)

TARGET_COMPONENTS = ["RECENT_CHECKUP", "GOT_FLUSHOT", "HAS_EXERCISE", "NON_SMOKER"]

# Only compute the score where ALL four components are non-null
all_present = df[TARGET_COMPONENTS].notna().all(axis=1)
df["PREV_HEALTH_SCORE"] = np.where(
    all_present,
    df[TARGET_COMPONENTS].sum(axis=1),
    np.nan,
)

df["PREV_HEALTH_HIGH"] = np.where(
    df["PREV_HEALTH_SCORE"].isna(), np.nan,
    np.where(df["PREV_HEALTH_SCORE"] >= 3, 1.0, 0.0),
)

print("Composite score distribution:")
print(df["PREV_HEALTH_SCORE"].value_counts(dropna=False).sort_index())
print(f"\nBinary target (PREV_HEALTH_HIGH):")
print(df["PREV_HEALTH_HIGH"].value_counts(dropna=False).sort_index())
print(f"\nRows with valid target: {df['PREV_HEALTH_HIGH'].notna().sum():,} / {len(df):,}"
      f" ({df['PREV_HEALTH_HIGH'].notna().mean()*100:.1f}%)")

In [ ]:

DEMO_FEATURES = ["_AGE_G", "_SEX", "_IMPRACE", "MARITAL"]

SOCIO_FEATURES = ["_EDUCAG", "INCOME_GROUP", "EMPLOY1"]

BEHAVIORAL_FEATURES = ["_BMI5_SCALED", "HAS_DEPRESSION"]

ACCESS_FEATURES = [
    "HAS_PERSONAL_DOCTOR",
    "HAS_HEALTH_PLAN", "HAS_HEALTH_PLAN_MISSING",
    "COST_BARRIER", "COST_BARRIER_MISSING",
]

GEO_FEATURES = ["STATE_FIPS"]

# Combined (all available predictors)
ALL_FEATURES = DEMO_FEATURES + SOCIO_FEATURES + BEHAVIORAL_FEATURES + ACCESS_FEATURES + GEO_FEATURES

print("Feature groups:")
print(f"  Demographic:       {DEMO_FEATURES}")
print(f"  Socioeconomic:     {SOCIO_FEATURES}")
print(f"  Behavioral/Health: {BEHAVIORAL_FEATURES}")
print(f"  Healthcare Access: {ACCESS_FEATURES}")
print(f"  Geography:         {GEO_FEATURES}")
print(f"\n  Total predictors:  {len(ALL_FEATURES)}")

In [ ]:
df_model = df[df["PREV_HEALTH_HIGH"].notna()].copy()
print(f"Modeling partition: {len(df_model):,} rows (dropped {len(df) - len(df_model):,} with missing target)")

y = df_model["PREV_HEALTH_HIGH"]

train_idx, test_idx = train_test_split(
    df_model.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

df_train = df_model.loc[train_idx].copy()
df_test = df_model.loc[test_idx].copy()

print(f"\nTrain: {len(df_train):,}  |  Test: {len(df_test):,}")
print(f"\nTarget distribution (train):")
print(df_train["PREV_HEALTH_HIGH"].value_counts(normalize=True).sort_index().round(4))
print(f"\nTarget distribution (test):")
print(df_test["PREV_HEALTH_HIGH"].value_counts(normalize=True).sort_index().round(4))

In [ ]:
checks = []

# 1. No overlap between train and test indices
overlap = set(train_idx) & set(test_idx)
checks.append(("No train/test overlap", len(overlap) == 0))

# 2. Stratification preserved class ratios
train_ratio = df_train["PREV_HEALTH_HIGH"].mean()
test_ratio = df_test["PREV_HEALTH_HIGH"].mean()
checks.append(("Class ratio preserved (within 0.5pp)", abs(train_ratio - test_ratio) < 0.005))

# 3. Sample weights are present and positive
checks.append(("Train weights all positive", (df_train["sample_weight"] > 0).all()))
checks.append(("Test weights all positive", (df_test["sample_weight"] > 0).all()))

# 4. Split ratio is approximately 80/20
actual_test_pct = len(df_test) / (len(df_train) + len(df_test))
checks.append(("Split ratio ~20% test", abs(actual_test_pct - 0.20) < 0.01))

# 5. All five survey years present in both partitions
checks.append(("All years in train", df_train["SURVEY_YEAR"].nunique() == 5))
checks.append(("All years in test", df_test["SURVEY_YEAR"].nunique() == 5))

print("Verification:")
for label, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label}")

In [ ]:
print("Missing values in training set (key columns):\n")
cols_check = ALL_FEATURES + TARGET_COMPONENTS + ["PREV_HEALTH_SCORE", "PREV_HEALTH_HIGH"]
missing = df_train[cols_check].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    for col, cnt in missing.items():
        print(f"  {col:30s}  {cnt:>8,}  ({cnt/len(df_train)*100:5.1f}%)")
else:
    print("  No missing values in key columns.")
print(f"\nTotal training rows: {len(df_train):,}")

In [ ]:

# Base model predictors (demographic + socioeconomic)
SQ2_BASE_FEATURES = DEMO_FEATURES + SOCIO_FEATURES

# Expanded model adds healthcare access
SQ2_EXPANDED_FEATURES = SQ2_BASE_FEATURES + ACCESS_FEATURES

print("SQ2 Base model features:")
print(f"  {SQ2_BASE_FEATURES}")
print(f"\nSQ2 Expanded model adds:")
print(f"  {ACCESS_FEATURES}")

sq2_train = df_train.copy()
sq2_test = df_test.copy()

for col in ["HAS_HEALTH_PLAN", "COST_BARRIER"]:
    train_mode = sq2_train.loc[sq2_train[col].notna(), col].mode().iloc[0]
    sq2_train[col] = sq2_train[col].fillna(train_mode)
    sq2_test[col] = sq2_test[col].fillna(train_mode)
    print(f"\n{col}: imputed missing with mode = {train_mode:.0f}")
    print(f"  Train non-null: {sq2_train[col].notna().sum():,}")
    print(f"  Test non-null:  {sq2_test[col].notna().sum():,}")

print("\nThe _MISSING indicator columns are still present so the model can")
print("distinguish 'imputed because not asked' from actual responses.")

In [ ]:
base_train_new = sq2_train[ ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1','PREV_HEALTH_HIGH']]

base_test_new = sq2_test[ ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1','PREV_HEALTH_HIGH']]

In [ ]:
sq2_train_enc = pd.get_dummies(base_train_new,columns = SQ2_BASE_FEATURES, drop_first = True,dtype = int)

sq2_test_enc = pd.get_dummies(base_test_new,columns = SQ2_BASE_FEATURES, drop_first = True, dtype = int)

In [ ]:
sq2_train_enc, sq2_test_enc = sq2_train_enc.align(
    sq2_test_enc,
    join="left",
    axis=1,
    fill_value=0
)

print("Train shape:", sq2_train_enc.shape)
print("Test shape:", sq2_test_enc.shape)

In [ ]:
import statsmodels.api as sm

X = sq2_train_enc.drop('PREV_HEALTH_HIGH',axis = 1)
y = sq2_train_enc['PREV_HEALTH_HIGH']

X = sm.add_constant(X)

model = sm.GLM(y,X,family=sm.families.Binomial(link=sm.families.links.Logit()), 
               freq_weights=sq2_train['sample_weight'])

result_base = model.fit()


print(result_base.summary())

In [ ]:
import numpy as np
np.exp(result_base.params)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_pred_base = result_base.predict(X)

y_true = y

fpr_base, tpr_base, _ = roc_curve(y_true, y_pred_base)
auc_base = auc(fpr_base, tpr_base)

In [ ]:
exp_train_new = sq2_train[ ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1','PREV_HEALTH_HIGH','HAS_HEALTH_PLAN', 'HAS_HEALTH_PLAN_MISSING','HAS_PERSONAL_DOCTOR', 'COST_BARRIER', 'COST_BARRIER_MISSING']].copy()
exp_test_new = sq2_test[ ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1','PREV_HEALTH_HIGH','HAS_HEALTH_PLAN', 'HAS_HEALTH_PLAN_MISSING','HAS_PERSONAL_DOCTOR', 'COST_BARRIER', 'COST_BARRIER_MISSING']].copy()



In [ ]:
SQ2_EXP_NEW = ['_AGE_G', '_SEX', '_IMPRACE', 'MARITAL', '_EDUCAG', 'INCOME_GROUP', 'EMPLOY1','HAS_HEALTH_PLAN', 'HAS_HEALTH_PLAN_MISSING',"HAS_PERSONAL_DOCTOR",
    "COST_BARRIER", "COST_BARRIER_MISSING"]



In [ ]:
sq2_exp_train_enc = pd.get_dummies(exp_train_new,columns = SQ2_EXP_NEW, drop_first = True,dtype = int)

sq2_exp_test_enc = pd.get_dummies(exp_test_new,columns = SQ2_EXP_NEW, drop_first = True, dtype = int)

In [ ]:
sq2_exp_train_enc, sq2_exp_test_enc = sq2_exp_train_enc.align(
    sq2_exp_test_enc,
    join="left",
    axis=1,
    fill_value=0
)

print("Train shape:", sq2_exp_train_enc.shape)
print("Test shape:", sq2_exp_test_enc.shape)

In [ ]:
X = sq2_exp_train_enc.drop('PREV_HEALTH_HIGH',axis = 1)
y = sq2_exp_train_enc['PREV_HEALTH_HIGH']

X = sm.add_constant(X)

exp_model = sm.GLM(y,X,family=sm.families.Binomial(link=sm.families.links.Logit()), 
               freq_weights=sq2_train['sample_weight'])

result_exp = exp_model.fit()


print(result_exp.summary())

In [ ]:
np.exp(result_exp.params)

In [ ]:
from scipy.stats import chi2

lr_stat = 2 * (result_exp.llf - result_base.llf)
df_diff = result_exp.df_model - result_base.df_model
p_value = chi2.sf(lr_stat, df_diff)

print("LR statistic:", lr_stat)
print("df difference:", df_diff)
print("p-value:", p_value)

In [ ]:
# Expanded model predictions
y_pred_exp = result_exp.predict(X)

# True labels
y_true = y

In [ ]:
fpr_exp, tpr_exp, _ = roc_curve(y_true, y_pred_exp)
auc_exp = auc(fpr_exp, tpr_exp)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(fpr_base, tpr_base, label=f'Base Model (AUC = {auc_base:.3f})')
plt.plot(fpr_exp, tpr_exp, label=f'Expanded Model (AUC = {auc_exp:.3f})')

# Diagonal line (random guess)
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Base vs Expanded Model')
plt.legend()
plt.grid()

plt.show()

In [ ]:
# Calculate AIC

delta_aic = result_base.aic - result_exp.aic

delta_aic

In [ ]:
import pandas as pd

model_comparison = pd.DataFrame({
    "Model": ["Base Model", "Expanded Model"],
    "Log-Likelihood": [result_base.llf, result_exp.llf],
    "AIC": [result_base.aic, result_exp.aic],
    "Pseudo R²": [0.1062, 0.1239]  # use values from your output
})

model_comparison["Log-Likelihood"] = model_comparison["Log-Likelihood"].round(2)
model_comparison["AIC"] = model_comparison["AIC"].round(2)
model_comparison["Pseudo R²"] = model_comparison["Pseudo R²"].round(4)

model_comparison

In [ ]:
comparison_tests = pd.DataFrame({
    "Metric": ["Likelihood Ratio Test", "df Difference", "p-value", "ΔAIC"],
    "Value": [lr_stat, df_diff, p_value, delta_aic]
})

comparison_tests["Value"] = comparison_tests["Value"].apply(
    lambda x: f"{x:.3f}" if x != 0 else "<0.001"
)

comparison_tests

In [ ]:
import numpy as np

access_vars = [
    "HAS_HEALTH_PLAN_1.0",
    "HAS_PERSONAL_DOCTOR_1.0",
    "COST_BARRIER_1.0"
]

odds_table = pd.DataFrame({
    "Variable": access_vars,
    "Odds Ratio": np.exp(result_exp.params[access_vars]),
    "p-value": result_exp.pvalues[access_vars]
})

odds_table["Odds Ratio"] = odds_table["Odds Ratio"].round(3)
odds_table["p-value"] = odds_table["p-value"].apply(lambda x: "<0.001" if x < 0.001 else round(x, 3))

odds_table["Interpretation"] = [
    "Higher odds of preventative behavior",
    "Higher odds of preventative behavior",
    "Lower odds of preventative behavior"
]

odds_table

In [ ]:

# Full feature set for SQ3 (all predictor families compete)
SQ3_FEATURES = ALL_FEATURES + ["SURVEY_YEAR"]

# Separate X, y, weights for train and test
X_train_sq3 = df_train[SQ3_FEATURES].copy()
y_train_sq3 = df_train["PREV_HEALTH_HIGH"].copy()
w_train_sq3 = df_train["sample_weight"].copy()

X_test_sq3 = df_test[SQ3_FEATURES].copy()
y_test_sq3 = df_test["PREV_HEALTH_HIGH"].copy()
w_test_sq3 = df_test["sample_weight"].copy()

# Per-behavior targets for secondary analysis
BEHAVIOR_TARGETS = {
    "checkup":  "RECENT_CHECKUP",
    "flushot":  "GOT_FLUSHOT",
    "exercise": "HAS_EXERCISE",
    "nonsmoker": "NON_SMOKER",
}

print("SQ3 ready:")
print(f"  Features: {len(SQ3_FEATURES)}")
print(f"  Train: {X_train_sq3.shape}")
print(f"  Test:  {X_test_sq3.shape}")
print(f"\n  Per-behavior targets available: {list(BEHAVIOR_TARGETS.keys())}")
print(f"\nNext steps: feature encoding, imputation, model fitting, tuning.")

In [ ]:
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss

warnings.filterwarnings('ignore') # Suppress target encoder/imputer warnings for clean output

# 1. Feature Classification based on Methodology
TARGET_ENCODE_COLS = ["STATE_FIPS"]
ORDINAL_COLS = ["_AGE_G", "_EDUCAG", "INCOME_GROUP"]
NOMINAL_COLS = ["_SEX", "_IMPRACE", "MARITAL", "EMPLOY1", "SURVEY_YEAR"]
NUMERIC_COLS = [col for col in SQ3_FEATURES if col not in (TARGET_ENCODE_COLS + ORDINAL_COLS + NOMINAL_COLS)]

# 2. Preprocessing Pipelines 
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

nominal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

target_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), 
    ("encoder", TargetEncoder(cols=[0])) 
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_COLS),
        ("ord", ordinal_transformer, ORDINAL_COLS),
        ("nom", nominal_transformer, NOMINAL_COLS),
        ("te", target_transformer, TARGET_ENCODE_COLS)
    ],
    remainder="drop"
)

# 3. Model Pipelines
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)) # <--- Set to -1
])

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric="auc"))
])

# 4. Hyperparameter Grids
rf_param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

xgb_param_grid = {
    "classifier__n_estimators": [200, 500, 1000],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__max_depth": [3, 5, 7, 9],
    "classifier__subsample": [0.7, 0.8, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 1.0]
}

# 5. Model Training & Tuning Setup
print("Setting up RandomizedSearchCV for Random Forest...")
rf_search = RandomizedSearchCV(
    rf_pipeline, 
    param_distributions=rf_param_grid, 
    n_iter=10,            
    cv=5,                
    scoring="roc_auc", 
    n_jobs=None,        
    random_state=RANDOM_STATE,
    verbose=3            
)

print("Setting up RandomizedSearchCV for XGBoost...")
xgb_search = RandomizedSearchCV(
    xgb_pipeline, 
    param_distributions=xgb_param_grid, 
    n_iter=10,            
    cv=5,                
    scoring="roc_auc", 
    n_jobs=None,         
    random_state=RANDOM_STATE,
    verbose=3            
)

# 6. Evaluation Function (AUC-ROC, AUC-PR, F1, Brier)
def evaluate_model(model, X_test, y_test, model_name):
    # Predict probabilities and classes
    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    
    # Calculate metrics
    auc_roc = roc_auc_score(y_test, probs)
    auc_pr = average_precision_score(y_test, probs)
    f1 = f1_score(y_test, preds)
    brier = brier_score_loss(y_test, probs)
    
    print(f"\\n--- {model_name} Holdout Performance ---")
    print(f"AUC-ROC:   {auc_roc:.4f}")
    print(f"AUC-PR:    {auc_pr:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Brier:     {brier:.4f}")
    print(f"Best Params: {model.best_params_}")

# Fit Random Forest (passing sample weights through the pipeline)
print("\\nFitting Random Forest...")
rf_search.fit(X_train_sq3, y_train_sq3, classifier__sample_weight=w_train_sq3)
evaluate_model(rf_search, X_test_sq3, y_test_sq3, "Random Forest")

print("\\nPre-processing evaluation set for XGBoost early stopping...")
X_test_transformed = preprocessor.fit_transform(X_test_sq3, y_test_sq3) 

print("Fitting XGBoost...")
xgb_search.fit(
    X_train_sq3, y_train_sq3,
    classifier__sample_weight=w_train_sq3,
    classifier__verbose=False
)
evaluate_model(xgb_search, X_test_sq3, y_test_sq3, "XGBoost")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Get the best XGBoost model from the search
best_xgb_pipeline = xgb_search.best_estimator_

# 2. Extract feature names from the preprocessor
feature_names = best_xgb_pipeline.named_steps["preprocessor"].get_feature_names_out()
clean_names = [name.split("__")[-1] for name in feature_names]

# 3. Extract importances from the XGBoost classifier
importances = best_xgb_pipeline.named_steps["classifier"].feature_importances_

# 4. Create a DataFrame and plot
df_importance = pd.DataFrame({"Feature": clean_names, "Importance": importances})
df_importance = df_importance.sort_values(by="Importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_importance, x="Importance", y="Feature", palette="viridis")
plt.title("Top 15 Predictors for Composite Preventive Health")
plt.xlabel("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()

In [ ]:
# Extract the best parameters we just found
best_xgb_params = xgb_search.best_params_

clean_params = {k.replace('classifier__', ''): v for k, v in best_xgb_params.items()}
# Add our standard params
clean_params['random_state'] = RANDOM_STATE
clean_params['n_jobs'] = -1
clean_params['eval_metric'] = 'auc'

secondary_results = {}

print("Training secondary models on individual behaviors...\n")

for behavior_name, target_col in BEHAVIOR_TARGETS.items():
    print(f"Modeling: {behavior_name} ({target_col})")
    
    # 1. Get the specific target data
    y_train_sec = df_train[target_col]
    y_test_sec = df_test[target_col]
    
    # Filter out rows where this specific behavior target might be NaN
    valid_train_idx = y_train_sec.notna()
    valid_test_idx = y_test_sec.notna()
    
    X_train_clean = X_train_sq3[valid_train_idx]
    y_train_clean = y_train_sec[valid_train_idx]
    w_train_clean = w_train_sq3[valid_train_idx]
    
    X_test_clean = X_test_sq3[valid_test_idx]
    y_test_clean = y_test_sec[valid_test_idx]
    
    # 2. Build a fresh pipeline using the best params
    sec_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(**clean_params))
    ])
    
    # 3. Fit the model
    sec_pipeline.fit(X_train_clean, y_train_clean, classifier__sample_weight=w_train_clean)
    
    # 4. Evaluate
    probs = sec_pipeline.predict_proba(X_test_clean)[:, 1]
    preds = sec_pipeline.predict(X_test_clean)
    
    secondary_results[behavior_name] = {
        "AUC-ROC": roc_auc_score(y_test_clean, probs),
        "AUC-PR": average_precision_score(y_test_clean, probs),
        "F1": f1_score(y_test_clean, preds),
        "Brier": brier_score_loss(y_test_clean, probs)
    }

print("\nSecondary models complete.")

In [ ]:
# Add our primary composite model to the results for a full comparison
secondary_results["COMPOSITE (Primary)"] = {
    "AUC-ROC": 0.7358,  
    "AUC-PR": 0.8578,   
    "F1": 0.8372,       
    "Brier": 0.1753     
}

# Generate the final table
df_comparison = pd.DataFrame(secondary_results).T
df_comparison = df_comparison[["AUC-ROC", "AUC-PR", "F1", "Brier"]] # Enforce column order

print("--- Final Model Performance Comparison ---")
display(df_comparison.round(4))

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))

# Plot Random Forest
RocCurveDisplay.from_estimator(
    rf_search.best_estimator_, X_test_sq3, y_test_sq3, 
    name="Random Forest", ax=ax, color="blue"
)

# Plot XGBoost
RocCurveDisplay.from_estimator(
    xgb_search.best_estimator_, X_test_sq3, y_test_sq3, 
    name="XGBoost", ax=ax, color="darkorange"
)

# Plot chance level
plt.plot([0, 1], [0, 1], "k--", label="Chance Level (AUC = 0.5)")

plt.title("ROC Curves on Holdout Set (Composite Target)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()
# Optional: plt.savefig("roc_curve.png", dpi=300)

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(8, 6))

# Plot Random Forest Calibration
CalibrationDisplay.from_estimator(
    rf_search.best_estimator_, X_test_sq3, y_test_sq3, 
    name="Random Forest", ax=ax, color="blue"
)

# Plot XGBoost Calibration
CalibrationDisplay.from_estimator(
    xgb_search.best_estimator_, X_test_sq3, y_test_sq3, 
    name="XGBoost", ax=ax, color="darkorange"
)

plt.title("Calibration Plots (Reliability Diagram)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()
# Optional: plt.savefig("calibration_plot.png", dpi=300)

In [ ]:
from sklearn.metrics import roc_auc_score

# 1. Clean up parameters for both models
best_rf_params = {k.replace('classifier__', ''): v for k, v in rf_search.best_params_.items()}
best_rf_params['random_state'] = RANDOM_STATE
best_rf_params['n_jobs'] = -1

best_xgb_params = {k.replace('classifier__', ''): v for k, v in xgb_search.best_params_.items()}
best_xgb_params['random_state'] = RANDOM_STATE
best_xgb_params['n_jobs'] = -1
best_xgb_params['eval_metric'] = 'auc'

behavior_results = []

print("Training secondary models on individual behaviors for BOTH algorithms...\n")

for behavior_name, target_col in BEHAVIOR_TARGETS.items():
    print(f"Modeling: {behavior_name} ({target_col})")
    
    # Filter valid rows for this specific target
    valid_train_idx = df_train[target_col].notna()
    valid_test_idx = df_test[target_col].notna()
    
    X_train_cl = X_train_sq3[valid_train_idx]
    y_train_cl = df_train[target_col][valid_train_idx]
    w_train_cl = w_train_sq3[valid_train_idx]
    
    X_test_cl = X_test_sq3[valid_test_idx]
    y_test_cl = df_test[target_col][valid_test_idx]
    
    rf_sec = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(**best_rf_params))
    ])
    rf_sec.fit(X_train_cl, y_train_cl, classifier__sample_weight=w_train_cl)
    rf_auc = roc_auc_score(y_test_cl, rf_sec.predict_proba(X_test_cl)[:, 1])
    
    xgb_sec = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(**best_xgb_params))
    ])
    xgb_sec.fit(X_train_cl, y_train_cl, classifier__sample_weight=w_train_cl)
    xgb_auc = roc_auc_score(y_test_cl, xgb_sec.predict_proba(X_test_cl)[:, 1])
    
    # Append results
    behavior_results.append({
        "Behavior": behavior_name, 
        "Random Forest AUC": rf_auc, 
        "XGBoost AUC": xgb_auc
    })

# Add Composite scores (from previous log) for complete table
behavior_results.append({
    "Behavior": "composite (primary)", 
    "Random Forest AUC": 0.7331, 
    "XGBoost AUC": 0.7358
})

# Format and Display Table
df_behavior_auc = pd.DataFrame(behavior_results).set_index("Behavior")
df_behavior_auc = df_behavior_auc.sort_values(by="XGBoost AUC", ascending=False)

print("\n--- AUC-ROC Comparison Across Behaviors ---")
display(df_behavior_auc.round(4))

In [ ]:

CATEGORICAL_FEATURES = [
    # Branch 1: Demographic / Socioeconomic
    ("_AGE_G",       7,  3, 1),   # 1-6 → remap to 1-6, 0=missing
    ("_SEX",         3,  1, 1),   # 1-2 → remap to 1-2, 0=missing
    ("_IMPRACE",     7,  3, 1),   # 1-6
    ("MARITAL",      8,  3, 1),   # 1-7 (including possible 9=refused mapped to missing)
    ("_EDUCAG",      5,  2, 1),   # 1-4
    ("INCOME_GROUP", 6,  3, 1),   # 1-5
    ("EMPLOY1",     10,  4, 1),   # 1-8 + possible 9
    ("STATE_FIPS",  55,  8, 1),   # ~54 unique states/territories
    # Branch 2: Access / Behavioral
    ("SURVEY_YEAR",  6,  3, 2),   # 2020-2024 → remap to 1-5
]

# Branch 2 continuous/binary features (no embedding needed)
BRANCH2_DENSE_FEATURES = [
    "HAS_PERSONAL_DOCTOR",
    "HAS_HEALTH_PLAN",
    "HAS_HEALTH_PLAN_MISSING",
    "COST_BARRIER",
    "COST_BARRIER_MISSING",
    "HAS_DEPRESSION",
    "_BMI5_SCALED",
]

print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"  Branch 1: {sum(1 for _,_,_,b in CATEGORICAL_FEATURES if b == 1)}")
print(f"  Branch 2: {sum(1 for _,_,_,b in CATEGORICAL_FEATURES if b == 2)}")
print(f"Branch 2 dense features: {len(BRANCH2_DENSE_FEATURES)}")
total_emb_params = sum(n * d for _, n, d, _ in CATEGORICAL_FEATURES)
print(f"Total embedding parameters: {total_emb_params:,}")

In [ ]:

def prepare_categorical_inputs(df_train, df_test, cat_features):
    """
    For each categorical feature, create integer-coded arrays suitable for
    Embedding layers. Missing values map to index 0.
    Returns dict of {col_name: (train_array, test_array)}.
    """
    cat_inputs = {}

    for col, n_cats, emb_dim, branch in cat_features:
        train_col = df_train[col].copy()
        test_col = df_test[col].copy()

        if col == "STATE_FIPS":
            # Remap FIPS codes to contiguous integers 1..N
            unique_fips = sorted(train_col.dropna().unique())
            fips_map = {fips: i + 1 for i, fips in enumerate(unique_fips)}
            train_col = train_col.map(fips_map)  # unmapped → NaN
            test_col = test_col.map(fips_map)

        elif col == "SURVEY_YEAR":
            # Remap 2020-2024 → 1-5
            year_map = {2020: 1, 2021: 2, 2022: 3, 2023: 4, 2024: 5}
            train_col = train_col.map(year_map)
            test_col = test_col.map(year_map)

        # Fill NaN with 0 (the "unknown" embedding index)
        train_arr = train_col.fillna(0).astype(np.int32).values
        test_arr = test_col.fillna(0).astype(np.int32).values

        # Clip to valid range [0, n_cats-1] to prevent index errors
        train_arr = np.clip(train_arr, 0, n_cats - 1)
        test_arr = np.clip(test_arr, 0, n_cats - 1)

        cat_inputs[col] = (train_arr, test_arr)
        print(f"  {col:20s}  unique(train): {len(np.unique(train_arr)):3d}  "
              f"range: [{train_arr.min()}, {train_arr.max()}]  "
              f"missing→0: {(train_arr == 0).sum():,}")

    return cat_inputs

print("Preparing categorical inputs:")
cat_inputs = prepare_categorical_inputs(df_train, df_test, CATEGORICAL_FEATURES)

In [ ]:

# Impute binary/categorical features with mode (fit on train only)
binary_cols = ["HAS_PERSONAL_DOCTOR", "HAS_HEALTH_PLAN", "COST_BARRIER", "HAS_DEPRESSION"]
imputer_mode = SimpleImputer(strategy="most_frequent")
df_train_b2 = df_train[BRANCH2_DENSE_FEATURES].copy()
df_test_b2 = df_test[BRANCH2_DENSE_FEATURES].copy()

df_train_b2[binary_cols] = imputer_mode.fit_transform(df_train_b2[binary_cols])
df_test_b2[binary_cols] = imputer_mode.transform(df_test_b2[binary_cols])

# Impute BMI with median + create missingness indicator
bmi_missing_train = df_train_b2["_BMI5_SCALED"].isna().astype(np.float32).values
bmi_missing_test = df_test_b2["_BMI5_SCALED"].isna().astype(np.float32).values

imputer_bmi = SimpleImputer(strategy="median")
df_train_b2[["_BMI5_SCALED"]] = imputer_bmi.fit_transform(df_train_b2[["_BMI5_SCALED"]])
df_test_b2[["_BMI5_SCALED"]] = imputer_bmi.transform(df_test_b2[["_BMI5_SCALED"]])

# Standardize BMI (the only truly continuous feature)
scaler_bmi = StandardScaler()
df_train_b2[["_BMI5_SCALED"]] = scaler_bmi.fit_transform(df_train_b2[["_BMI5_SCALED"]])
df_test_b2[["_BMI5_SCALED"]] = scaler_bmi.transform(df_test_b2[["_BMI5_SCALED"]])

# Add BMI missingness indicator
df_train_b2["_BMI5_MISSING"] = bmi_missing_train
df_test_b2["_BMI5_MISSING"] = bmi_missing_test

# Convert to float32 arrays
x_dense_train = df_train_b2.values.astype(np.float32)
x_dense_test = df_test_b2.values.astype(np.float32)
dense_col_names = list(df_train_b2.columns)

print(f"Dense Branch 2 features: {x_dense_train.shape[1]}")
print(f"  Columns: {dense_col_names}")
print(f"  NaN check — train: {np.isnan(x_dense_train).any()}, test: {np.isnan(x_dense_test).any()}")
print(f"  BMI missing (train): {bmi_missing_train.sum():,} ({bmi_missing_train.mean()*100:.1f}%)")

In [ ]:

def make_input_dict(cat_inputs, dense_arr, split="train"):
    """Build the {input_name: array} dict for model.fit() / model.predict()."""
    idx = 0 if split == "train" else 1
    d = {}
    for col, n_cats, emb_dim, branch in CATEGORICAL_FEATURES:
        d[f"cat_{col}"] = cat_inputs[col][idx]
    d["dense_branch2"] = dense_arr
    return d

X_train_dict = make_input_dict(cat_inputs, x_dense_train, "train")
X_test_dict = make_input_dict(cat_inputs, x_dense_test, "test")

# Targets and weights
y_train = df_train["PREV_HEALTH_HIGH"].values.astype(np.float32)
y_test = df_test["PREV_HEALTH_HIGH"].values.astype(np.float32)
w_train = df_train["sample_weight"].values.astype(np.float32)

print("Model input dict keys:")
for k, v in X_train_dict.items():
    print(f"  {k:25s}  shape: {v.shape}  dtype: {v.dtype}")

In [ ]:
def build_embedding_dual_path(
    cat_features=CATEGORICAL_FEATURES,
    n_dense_b2=8,  # number of dense Branch 2 features
    b1_width=128,
    b1_layers=2,
    b2_width=64,
    b2_layers=2,
    head_width=128,
    head_layers=2,
    dropout=0.3,
    l2_reg=1e-5,
    learning_rate=1e-3,
):
    """
    Build a dual-path binary classifier with entity embeddings for categoricals.

    Branch 1 receives all demographic/socioeconomic categoricals as embeddings.
    Branch 2 receives healthcare access + behavioral features (dense) plus
    SURVEY_YEAR as an embedding.

    Returns: compiled Keras Model
    """
    reg = l2(l2_reg) if l2_reg > 0 else None

    cat_input_layers = {}   # col_name → Input layer
    branch1_embeddings = []
    branch2_embeddings = []

    for col, n_cats, emb_dim, branch in cat_features:
        inp = Input(shape=(1,), name=f"cat_{col}", dtype="int32")
        cat_input_layers[col] = inp
        emb = Embedding(
            input_dim=n_cats,
            output_dim=emb_dim,
            name=f"emb_{col}",
        )(inp)
        flat = Flatten(name=f"flat_{col}")(emb)

        if branch == 1:
            branch1_embeddings.append(flat)
        else:
            branch2_embeddings.append(flat)

    b1 = Concatenate(name="b1_concat")(branch1_embeddings)
    for i in range(b1_layers):
        b1 = Dense(b1_width, kernel_regularizer=reg, name=f"b1_dense_{i}")(b1)
        b1 = BatchNormalization(name=f"b1_bn_{i}")(b1)
        b1 = Activation("relu", name=f"b1_relu_{i}")(b1)
        b1 = Dropout(dropout, name=f"b1_drop_{i}")(b1)

    dense_input = Input(shape=(n_dense_b2,), name="dense_branch2", dtype="float32")
    b2_parts = branch2_embeddings + [dense_input]
    b2 = Concatenate(name="b2_concat")(b2_parts)
    for i in range(b2_layers):
        b2 = Dense(b2_width, kernel_regularizer=reg, name=f"b2_dense_{i}")(b2)
        b2 = BatchNormalization(name=f"b2_bn_{i}")(b2)
        b2 = Activation("relu", name=f"b2_relu_{i}")(b2)
        b2 = Dropout(dropout, name=f"b2_drop_{i}")(b2)

    merged = Concatenate(name="merge")([b1, b2])
    h = merged
    width = head_width
    for i in range(head_layers):
        h = Dense(width, kernel_regularizer=reg, name=f"head_dense_{i}")(h)
        h = BatchNormalization(name=f"head_bn_{i}")(h)
        h = Activation("relu", name=f"head_relu_{i}")(h)
        h = Dropout(dropout, name=f"head_drop_{i}")(h)
        width = max(width // 2, 32)  # taper the head

    output = Dense(1, activation="sigmoid", name="preventive_health")(h)

    all_inputs = [cat_input_layers[col] for col, *_ in cat_features] + [dense_input]
    model = Model(inputs=all_inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[AUC(name="auc"), "accuracy"],
    )

    return model

In [ ]:
model = build_embedding_dual_path(n_dense_b2=x_dense_train.shape[1])
model.summary()

In [ ]:
nn_train_idx, nn_val_idx = train_test_split(
    np.arange(len(y_train)),
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

# Split each input array
X_fit = {k: v[nn_train_idx] for k, v in X_train_dict.items()}
X_val = {k: v[nn_val_idx] for k, v in X_train_dict.items()}
y_fit = y_train[nn_train_idx]
y_val = y_train[nn_val_idx]
w_fit = w_train[nn_train_idx]
w_val = w_train[nn_val_idx]

print(f"Fit:  {len(y_fit):,}  (positive rate: {y_fit.mean():.4f})")
print(f"Val:  {len(y_val):,}  (positive rate: {y_val.mean():.4f})")
print(f"Test: {len(y_test):,}  (positive rate: {y_test.mean():.4f})")

In [ ]:

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0.0, 1.0])
cw = compute_class_weight("balanced", classes=classes, y=y_fit)
class_weight_map = {0: cw[0], 1: cw[1]}
print(f"Class weights: {class_weight_map}")

# Multiply: survey_weight * class_weight_for_that_row's_label
w_fit_adj = w_fit * np.where(y_fit == 1, class_weight_map[1], class_weight_map[0])
w_val_adj = w_val * np.where(y_val == 1, class_weight_map[1], class_weight_map[0])

print(f"Adjusted weight stats (fit):  mean={w_fit_adj.mean():.4f}  std={w_fit_adj.std():.4f}")
print(f"Adjusted weight stats (val):  mean={w_val_adj.mean():.4f}  std={w_val_adj.std():.4f}")

callbacks = [
    EarlyStopping(
        monitor="val_auc", mode="max", patience=15,
        restore_best_weights=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_auc", mode="max", factor=0.5,
        patience=5, min_lr=1e-6, verbose=1,
    ),
    ModelCheckpoint(
        filepath=str(OUT_DIR / "nn_embedding_best.keras"),
        monitor="val_auc", mode="max",
        save_best_only=True, verbose=1,
    ),
]

In [ ]:
history = model.fit(
    X_fit, y_fit,
    sample_weight=w_fit_adj,
    validation_data=(X_val, y_val, w_val_adj),
    epochs=60,
    batch_size=1024,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
y_prob = model.predict(X_test_dict, batch_size=2048, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
auc_roc = roc_auc_score(y_test, y_prob)
auc_pr = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"  AUC-ROC:          {auc_roc:.4f}")
print(f"  AUC-PR:           {auc_pr:.4f}")
print(f"  Brier Score:      {brier:.4f}")
print(f"  F1 Score:         {f1:.4f}")
print(f"  Accuracy:         {(y_pred == y_test).mean():.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low (0-2)", "High (3-4)"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# AUC
axes[0].plot(history.history["auc"], label="Train AUC")
axes[0].plot(history.history["val_auc"], label="Val AUC")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("AUC")
axes[0].set_title("AUC-ROC")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Binary Cross-Entropy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
if "lr" in history.history:
    axes[2].plot(history.history["lr"])
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("LR Schedule")
    axes[2].set_yscale("log")
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "LR history\nnot available", ha="center", va="center")

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_training_history.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="NN (Embeddings)")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title(f"ROC Curve (AUC = {auc_roc:.4f})")
axes[0].grid(True, alpha=0.3)

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="NN (Embeddings)")
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", alpha=0.3, label="Baseline")
axes[1].set_title(f"Precision-Recall Curve (AP = {auc_pr:.4f})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import keras_tuner as kt

def build_tunable_model(hp):
    """Keras Tuner model builder — searches over architecture hyperparameters."""
    return build_embedding_dual_path(
        n_dense_b2=x_dense_train.shape[1],
        b1_width=hp.Choice("b1_width", [64, 128, 256]),
        b1_layers=hp.Int("b1_layers", 1, 3),
        b2_width=hp.Choice("b2_width", [32, 64, 128]),
        b2_layers=hp.Int("b2_layers", 1, 2),
        head_width=hp.Choice("head_width", [64, 128, 256]),
        head_layers=hp.Int("head_layers", 1, 3),
        dropout=hp.Float("dropout", 0.15, 0.5, step=0.05),
        l2_reg=hp.Choice("l2_reg", [0.0, 1e-5, 1e-4]),
        learning_rate=hp.Choice("lr", [3e-4, 1e-3, 3e-3]),
    )

tuner = kt.Hyperband(
    build_tunable_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_epochs=30,
    factor=3,
    hyperband_iterations=2,
    directory=str(OUT_DIR / "kt_logs"),
    project_name="nn_embedding_search",
    overwrite=True,
    seed=RANDOM_STATE,
)

tuner.search_space_summary()

In [ ]:
tuner_callbacks = [
    EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=3, min_lr=1e-6),
]

tuner.search(
    X_fit, y_fit,
    sample_weight=w_fit_adj,
    validation_data=(X_val, y_val, w_val_adj),
    batch_size=1024,
    callbacks=tuner_callbacks,
    verbose=2,
)

print("\nBest hyperparameters:")
best_hp = tuner.get_best_hyperparameters(1)[0]
for param in ["b1_width", "b1_layers", "b2_width", "b2_layers",
              "head_width", "head_layers", "dropout", "l2_reg", "lr"]:
    print(f"  {param}: {best_hp.get(param)}")

In [ ]:
best_model = tuner.get_best_models(1)[0]

# Evaluate the tuned model on test set
y_prob_tuned = best_model.predict(X_test_dict, batch_size=2048, verbose=0).ravel()
y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)

auc_roc_tuned = roc_auc_score(y_test, y_prob_tuned)
auc_pr_tuned = average_precision_score(y_test, y_prob_tuned)
brier_tuned = brier_score_loss(y_test, y_prob_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)

print("=" * 50)
print("TUNED MODEL — TEST SET EVALUATION")
print("=" * 50)
print(f"  AUC-ROC:          {auc_roc_tuned:.4f}  (baseline: {auc_roc:.4f})")
print(f"  AUC-PR:           {auc_pr_tuned:.4f}  (baseline: {auc_pr:.4f})")
print(f"  Brier Score:      {brier_tuned:.4f}  (baseline: {brier:.4f})")
print(f"  F1 Score:         {f1_tuned:.4f}  (baseline: {f1:.4f})")
print()
print(classification_report(y_test, y_pred_tuned, target_names=["Low (0-2)", "High (3-4)"]))

In [ ]:
eval_model = best_model if "best_model" in dir() else model

# Branch 1 categorical features vs Branch 2
b1_cat_cols = [col for col, _, _, b in CATEGORICAL_FEATURES if b == 1]
b2_cat_cols = [col for col, _, _, b in CATEGORICAL_FEATURES if b == 2]

def ablated_predict(model, X_dict, zero_branch):
    """
    Predict with one branch zeroed out.
    zero_branch=1 → zero all Branch 1 (demo/socio) inputs
    zero_branch=2 → zero all Branch 2 (access/behavioral) inputs
    """
    X_ablated = {}
    for k, v in X_dict.items():
        if zero_branch == 1:
            # Zero out Branch 1 categoricals
            if any(k == f"cat_{col}" for col in b1_cat_cols):
                X_ablated[k] = np.zeros_like(v)  # maps to the "missing" embedding
            else:
                X_ablated[k] = v
        elif zero_branch == 2:
            # Zero out Branch 2 categoricals + dense
            if any(k == f"cat_{col}" for col in b2_cat_cols):
                X_ablated[k] = np.zeros_like(v)
            elif k == "dense_branch2":
                X_ablated[k] = np.zeros_like(v)
            else:
                X_ablated[k] = v
    return model.predict(X_ablated, batch_size=2048, verbose=0).ravel()

# Full model predictions
y_prob_full = eval_model.predict(X_test_dict, batch_size=2048, verbose=0).ravel()

# Branch 2 only (zero out Branch 1)
y_prob_b2_only = ablated_predict(eval_model, X_test_dict, zero_branch=1)

# Branch 1 only (zero out Branch 2)
y_prob_b1_only = ablated_predict(eval_model, X_test_dict, zero_branch=2)

auc_full = roc_auc_score(y_test, y_prob_full)
auc_b1_only = roc_auc_score(y_test, y_prob_b1_only)
auc_b2_only = roc_auc_score(y_test, y_prob_b2_only)

print("Branch Ablation Results (AUC-ROC on test set):")
print(f"  Full model (both branches):    {auc_full:.4f}")
print(f"  Branch 1 only (Demo/Socio):    {auc_b1_only:.4f}  (drop: {auc_full - auc_b1_only:+.4f})")
print(f"  Branch 2 only (Access/Behav):  {auc_b2_only:.4f}  (drop: {auc_full - auc_b2_only:+.4f})")
print(f"\n  Branch 1 contribution: {auc_full - auc_b2_only:.4f} AUC points")
print(f"  Branch 2 contribution: {auc_full - auc_b1_only:.4f} AUC points")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Full Model\n(Both Branches)", "Branch 1 Only\n(Demo/Socio)", "Branch 2 Only\n(Access/Behav)"]
aucs = [auc_full, auc_b1_only, auc_b2_only]
colors = ["#2ecc71", "#3498db", "#e74c3c"]
bars = ax.bar(labels, aucs, color=colors, edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("AUC-ROC")
ax.set_title("Branch Ablation: Contribution of Each Input Branch")
ax.set_ylim(0.5, max(aucs) + 0.03)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.3, label="Random")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_branch_ablation.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shap

N_SHAP = 500
N_BG = 100
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(y_test), size=N_SHAP, replace=False)
bg_idx = np.random.choice(len(y_train), size=N_BG, replace=False)

input_keys = list(X_test_dict.keys())

def stack_inputs(X_dict, idx):
    """Horizontally stack all input arrays into one float32 matrix."""
    parts = []
    for k in input_keys:
        arr = X_dict[k][idx]
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)
        parts.append(arr.astype(np.float32))
    return np.hstack(parts)

X_shap_flat = stack_inputs(X_test_dict, shap_idx)
X_bg_flat = stack_inputs(X_train_dict, bg_idx)

# Column boundaries for splitting back into the dict
col_splits = []
for k in input_keys:
    sample = X_test_dict[k][:1]
    col_splits.append(1 if sample.ndim == 1 else sample.shape[1] if sample.ndim == 2 else 1)

def predict_from_flat(X_flat):
    """Split a flat array back into a LIST of arrays (matching model.inputs order)."""
    inputs_list = []
    col = 0
    for k, width in zip(input_keys, col_splits):
        chunk = X_flat[:, col:col + width]
        if "cat_" in k:
            inputs_list.append(chunk.astype(np.int32))
        else:
            inputs_list.append(chunk.astype(np.float32))
        col += width
    return eval_model.predict(inputs_list, batch_size=2048, verbose=0).ravel()

# Sanity check: wrapper matches direct prediction
direct_inputs = [X_test_dict[k][shap_idx[:5]].reshape(-1, 1)
                 if X_test_dict[k][shap_idx[:5]].ndim == 1
                 else X_test_dict[k][shap_idx[:5]]
                 for k in input_keys]
direct = eval_model.predict(direct_inputs, batch_size=5, verbose=0).ravel()
wrapped = predict_from_flat(X_shap_flat[:5])
assert np.allclose(direct, wrapped, atol=1e-5), "Wrapper mismatch!"
print("Sanity check passed.")

# Feature names (one per column in the flat matrix)
feature_names = []
for k, width in zip(input_keys, col_splits):
    if k == "dense_branch2":
        feature_names.extend(dense_col_names)
    else:
        feature_names.append(k.replace("cat_", ""))

print(f"SHAP sample: {N_SHAP} test rows, {N_BG} background rows")
print(f"Flat input shape: {X_shap_flat.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")

In [ ]:

explainer = shap.KernelExplainer(predict_from_flat, X_bg_flat)
shap_values_raw = explainer.shap_values(X_shap_flat, nsamples=200)

shap_matrix = shap_values_raw
print(f"SHAP matrix shape: {shap_matrix.shape}")
print(f"Feature names ({len(feature_names)}): {feature_names}")

In [ ]:
mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(feature_names))
ax.barh(y_pos, mean_abs_shap[sorted_idx[::-1]],
        color="#3498db", edgecolor="black", linewidth=0.3)
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in sorted_idx[::-1]])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Importance — Neural Network (Entity Embeddings)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_shap_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print ranked importance
print("\nFeature importance ranking (mean |SHAP|):")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank:2d}. {feature_names[idx]:30s}  {mean_abs_shap[idx]:.5f}")

In [ ]:
from sklearn.decomposition import PCA

def get_embedding_weights(model, layer_name):
    """Extract the learned embedding matrix from a named layer."""
    return model.get_layer(layer_name).get_weights()[0]

# STATE_FIPS embedding (the most interesting one — 54 states → 8 dims)
state_emb = get_embedding_weights(eval_model, "emb_STATE_FIPS")
print(f"STATE_FIPS embedding shape: {state_emb.shape}")

# PCA to 2D for visualization
pca = PCA(n_components=2)
state_2d = pca.fit_transform(state_emb[1:])  # skip index 0 (missing bucket)

# FIPS code lookup for labels
unique_fips = sorted(df_train["STATE_FIPS"].dropna().unique())
fips_to_abbr = {
    1:"AL",2:"AK",4:"AZ",5:"AR",6:"CA",8:"CO",9:"CT",10:"DE",11:"DC",12:"FL",
    13:"GA",15:"HI",16:"ID",17:"IL",18:"IN",19:"IA",20:"KS",21:"KY",22:"LA",
    23:"ME",24:"MD",25:"MA",26:"MI",27:"MN",28:"MS",29:"MO",30:"MT",31:"NE",
    32:"NV",33:"NH",34:"NJ",35:"NM",36:"NY",37:"NC",38:"ND",39:"OH",40:"OK",
    41:"OR",42:"PA",44:"RI",45:"SC",46:"SD",47:"TN",48:"TX",49:"UT",50:"VT",
    51:"VA",53:"WA",54:"WV",55:"WI",56:"WY",66:"GU",72:"PR",78:"VI",
}

fig, ax = plt.subplots(figsize=(12, 8))
for i, fips in enumerate(unique_fips):
    if i < len(state_2d):
        abbr = fips_to_abbr.get(int(fips), str(int(fips)))
        ax.scatter(state_2d[i, 0], state_2d[i, 1], s=30, zorder=5)
        ax.annotate(abbr, (state_2d[i, 0], state_2d[i, 1]),
                    fontsize=7, ha="center", va="bottom")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Learned STATE_FIPS Embeddings (PCA projection)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_state_embeddings.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

emb_configs = [
    ("emb__AGE_G", "_AGE_G", {1:"18-24",2:"25-34",3:"35-44",4:"45-54",5:"55-64",6:"65+"}),
    ("emb_EMPLOY1", "EMPLOY1", {1:"Employed",2:"Self-emp",3:"Unemp>1y",4:"Unemp<1y",
                                 5:"Homemaker",6:"Student",7:"Retired",8:"Unable"}),
    ("emb__IMPRACE", "_IMPRACE", {1:"White",2:"Black",3:"Asian",4:"AI/AN",5:"Hispanic",6:"Other"}),
]

for ax, (layer_name, col, label_map) in zip(axes, emb_configs):
    emb_w = get_embedding_weights(eval_model, layer_name)
    # Skip index 0 (missing bucket), plot indices 1..N
    n_cats = len(label_map)
    emb_sub = emb_w[1:n_cats+1]

    if emb_sub.shape[1] >= 2:
        ax.scatter(emb_sub[:, 0], emb_sub[:, 1], s=60, zorder=5)
        for i, (code, label) in enumerate(sorted(label_map.items())):
            if i < len(emb_sub):
                ax.annotate(label, (emb_sub[i, 0], emb_sub[i, 1]),
                            fontsize=7, ha="center", va="bottom")
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")
    else:
        # 1D embedding — plot as number line
        vals = emb_sub.ravel()
        ax.scatter(vals, np.zeros_like(vals), s=60, zorder=5)
        for i, (code, label) in enumerate(sorted(label_map.items())):
            if i < len(vals):
                ax.annotate(label, (vals[i], 0), fontsize=7,
                            ha="center", va="bottom", rotation=45)
        ax.set_xlabel("Embedding value")
        ax.set_yticks([])

    ax.set_title(f"{col} Embedding")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / "nn_embeddings_other.png"), dpi=150, bbox_inches="tight")
plt.show()

all four models, same holdout.

In [ ]:
import pandas as pd

X_test_base = sq2_test_enc.drop('PREV_HEALTH_HIGH', axis=1)
X_test_base = sm.add_constant(X_test_base)
y_pred_base_test = result_base.predict(X_test_base)
auc_base_test = roc_auc_score(sq2_test_enc['PREV_HEALTH_HIGH'], y_pred_base_test)

X_test_exp = sq2_exp_test_enc.drop('PREV_HEALTH_HIGH', axis=1)
X_test_exp = sm.add_constant(X_test_exp)
y_pred_exp_test = result_exp.predict(X_test_exp)
auc_exp_test = roc_auc_score(sq2_exp_test_enc['PREV_HEALTH_HIGH'], y_pred_exp_test)

# RF and XGBoost from SQ3
rf_probs = rf_search.best_estimator_.predict_proba(X_test_sq3)[:, 1]
xgb_probs = xgb_search.best_estimator_.predict_proba(X_test_sq3)[:, 1]

rf_preds = rf_search.best_estimator_.predict(X_test_sq3)
xgb_preds = xgb_search.best_estimator_.predict(X_test_sq3)

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss

results = {
    "Logistic (Base)": {
        "AUC-ROC": auc_base_test,
    },
    "Logistic (Expanded)": {
        "AUC-ROC": auc_exp_test,
    },
    "Random Forest": {
        "AUC-ROC": roc_auc_score(y_test_sq3, rf_probs),
        "AUC-PR": average_precision_score(y_test_sq3, rf_probs),
        "F1": f1_score(y_test_sq3, rf_preds),
        "Brier": brier_score_loss(y_test_sq3, rf_probs),
    },
    "XGBoost": {
        "AUC-ROC": roc_auc_score(y_test_sq3, xgb_probs),
        "AUC-PR": average_precision_score(y_test_sq3, xgb_probs),
        "F1": f1_score(y_test_sq3, xgb_preds),
        "Brier": brier_score_loss(y_test_sq3, xgb_probs),
    },
    "Neural Network (Embeddings)": {
        "AUC-ROC": auc_roc_tuned,
        "AUC-PR": auc_pr_tuned,
        "F1": f1_tuned,
        "Brier": brier_tuned,
    },
}

df_results = pd.DataFrame(results).T
print("=" * 60)
print("CROSS-MODEL HOLDOUT PERFORMANCE COMPARISON")
print("=" * 60)
display(df_results.round(4))

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_curve
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))

# SQ2 Logistic (Expanded)
fpr_log, tpr_log, _ = roc_curve(sq2_exp_test_enc['PREV_HEALTH_HIGH'], y_pred_exp_test)
ax.plot(fpr_log, tpr_log, label=f"Logistic Expanded (AUC={auc_exp_test:.4f})", linewidth=1.5)

# Random Forest
RocCurveDisplay.from_estimator(
    rf_search.best_estimator_, X_test_sq3, y_test_sq3,
    name="Random Forest", ax=ax
)

# XGBoost
RocCurveDisplay.from_estimator(
    xgb_search.best_estimator_, X_test_sq3, y_test_sq3,
    name="XGBoost", ax=ax
)

# Neural Network
fpr_nn, tpr_nn, _ = roc_curve(y_test, y_prob_tuned)
ax.plot(fpr_nn, tpr_nn, label=f"NN Embeddings (AUC={auc_roc_tuned:.4f})", linewidth=1.5)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Chance")
ax.set_title("ROC Curves — All Models on Shared Holdout Set")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "cross_model_roc.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shap

# TreeExplainer for RF and XGBoost (fast, exact)
print("Computing SHAP for Random Forest...")
rf_model = rf_search.best_estimator_
X_test_transformed_rf = rf_model.named_steps["preprocessor"].transform(X_test_sq3)
rf_feature_names = [n.split("__")[-1] for n in rf_model.named_steps["preprocessor"].get_feature_names_out()]

rf_explainer = shap.TreeExplainer(rf_model.named_steps["classifier"])
# Use a subsample for consistency
np.random.seed(RANDOM_STATE)
shap_idx_tree = np.random.choice(X_test_transformed_rf.shape[0], size=500, replace=False)
rf_shap_values = rf_explainer.shap_values(X_test_transformed_rf[shap_idx_tree])
if isinstance(rf_shap_values, list):
    rf_shap_values = rf_shap_values[1]  # class 1

print("Computing SHAP for XGBoost...")
xgb_model = xgb_search.best_estimator_
X_test_transformed_xgb = xgb_model.named_steps["preprocessor"].transform(X_test_sq3)
xgb_feature_names = [n.split("__")[-1] for n in xgb_model.named_steps["preprocessor"].get_feature_names_out()]

xgb_explainer = shap.TreeExplainer(xgb_model.named_steps["classifier"])
xgb_shap_values = xgb_explainer.shap_values(X_test_transformed_xgb[shap_idx_tree])

print(f"RF SHAP shape: {rf_shap_values.shape}")
print(f"XGBoost SHAP shape: {xgb_shap_values.shape}")
print(f"NN SHAP shape: {shap_matrix.shape}")

In [ ]:

ORIGINAL_FEATURES = [
    "_AGE_G", "_SEX", "_IMPRACE", "MARITAL", "_EDUCAG", "INCOME_GROUP", 
    "EMPLOY1", "STATE_FIPS", "SURVEY_YEAR",
    "HAS_PERSONAL_DOCTOR", "HAS_HEALTH_PLAN", "HAS_HEALTH_PLAN_MISSING",
    "COST_BARRIER", "COST_BARRIER_MISSING", "HAS_DEPRESSION",
    "_BMI5_SCALED", "_BMI5_MISSING"
]

def aggregate_shap_to_original(shap_vals, feature_names, original_features):
    """Sum |SHAP| for one-hot columns back to their parent feature."""
    import re
    mean_abs = np.abs(shap_vals).mean(axis=0)
    aggregated = {}
    for orig in original_features:
        # Find columns that start with this feature name
        mask = [f == orig or f.startswith(orig + "_") for f in feature_names]
        if any(mask):
            aggregated[orig] = mean_abs[mask].sum()
    # Also check for missingindicator columns
    for i, f in enumerate(feature_names):
        if 'missingindicator' in f.lower():
            # Try to map back to parent
            parent = f.replace('missingindicator_', '').split('_')[0]
            # Add to the parent feature if it exists, otherwise skip
            pass
    return aggregated

rf_agg = aggregate_shap_to_original(rf_shap_values, rf_feature_names, ORIGINAL_FEATURES)
xgb_agg = aggregate_shap_to_original(xgb_shap_values, xgb_feature_names, ORIGINAL_FEATURES)

# NN SHAP is already at the original feature level
nn_shap_agg = dict(zip(feature_names, np.abs(shap_matrix).mean(axis=0)))

# Combine into a comparison table
all_feats = sorted(set(list(rf_agg.keys()) + list(xgb_agg.keys()) + list(nn_shap_agg.keys())))
comparison_data = []
for feat in all_feats:
    comparison_data.append({
        "Feature": feat,
        "RF": rf_agg.get(feat, 0),
        "XGBoost": xgb_agg.get(feat, 0),
        "NN": nn_shap_agg.get(feat, 0),
    })

df_shap_compare = pd.DataFrame(comparison_data).set_index("Feature")

# Add rank columns
for col in ["RF", "XGBoost", "NN"]:
    df_shap_compare[f"{col}_rank"] = df_shap_compare[col].rank(ascending=False).astype(int)

# Sort by average rank
df_shap_compare["Avg_Rank"] = df_shap_compare[["RF_rank", "XGBoost_rank", "NN_rank"]].mean(axis=1)
df_shap_compare = df_shap_compare.sort_values("Avg_Rank")

print("=" * 70)
print("CROSS-MODEL SHAP FEATURE IMPORTANCE (mean |SHAP|)")
print("=" * 70)
display(df_shap_compare.round(5))

In [ ]:
import matplotlib.pyplot as plt

top_n = 10
df_top = df_shap_compare.head(top_n)[["RF", "XGBoost", "NN"]].copy()

# Normalize each model's SHAP to [0,1] for visual comparability
for col in ["RF", "XGBoost", "NN"]:
    max_val = df_shap_compare[col].max()
    if max_val > 0:
        df_top[col] = df_top[col] / max_val

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_top))
width = 0.25

bars1 = ax.bar(x - width, df_top["RF"], width, label="Random Forest", color="#3498db", edgecolor="black", linewidth=0.3)
bars2 = ax.bar(x, df_top["XGBoost"], width, label="XGBoost", color="#e67e22", edgecolor="black", linewidth=0.3)
bars3 = ax.bar(x + width, df_top["NN"], width, label="Neural Network", color="#2ecc71", edgecolor="black", linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(df_top.index, rotation=45, ha="right")
ax.set_ylabel("Normalized Mean |SHAP|")
ax.set_title("Top 10 Features by Average SHAP Rank Across Models")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "cross_model_shap_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# Print consensus ranking
print("\nConsensus Feature Ranking (by average rank across RF, XGBoost, NN):")
for i, (feat, row) in enumerate(df_shap_compare.head(top_n).iterrows(), 1):
    print(f"  {i:2d}. {feat:30s}  RF={int(row['RF_rank']):2d}  XGB={int(row['XGBoost_rank']):2d}  NN={int(row['NN_rank']):2d}  Avg={row['Avg_Rank']:.1f}")

In [ ]:
nn_preds = pd.DataFrame({
    "y_true": y_test,
    "y_prob_nn": y_prob_full,
    "y_pred_nn": (y_prob_full >= 0.5).astype(int),
}, index=df_test.index)
nn_preds.to_parquet(OUT_DIR / "nn_test_predictions.parquet")

shap_df = pd.DataFrame(shap_matrix, columns=feature_names)
shap_df.to_parquet(OUT_DIR / "nn_shap_values.parquet")

eval_model.save(str(OUT_DIR / "nn_embedding_final.keras"))

print("Saved outputs:")
print(f"  {OUT_DIR}/nn_test_predictions.parquet  ({len(nn_preds):,} rows)")
print(f"  {OUT_DIR}/nn_shap_values.parquet       ({shap_df.shape})")
print(f"  {OUT_DIR}/nn_embedding_final.keras")
print(f"  {OUT_DIR}/nn_training_history.png")
print(f"  {OUT_DIR}/nn_roc_pr_curves.png")
print(f"  {OUT_DIR}/nn_branch_ablation.png")
print(f"  {OUT_DIR}/nn_shap_importance.png")
print(f"  {OUT_DIR}/nn_state_embeddings.png")
print(f"  {OUT_DIR}/nn_embeddings_other.png")

print(f"\n{'='*50}")
print(f"FINAL METRICS SUMMARY")
print(f"{'='*50}")
print(f"  Default model AUC-ROC: {auc_roc:.4f}")
if "auc_roc_tuned" in dir():
    print(f"  Tuned model AUC-ROC:   {auc_roc_tuned:.4f}")
print(f"  Baseline notebook AUC: ~0.736 (val, from training logs)")